# Federated learning with Flower (simulation)

Five clients train together a model that recognises clothes and shoes (Fashion-MNIST,
10 classes), **without ever sharing their images**. In each round, the server sends the
model to the clients, each one trains it on its own data and sends back only the
weights, then the server averages them.

Here the five clients are simulated on this machine, each in its own process with its
own data. The app is the same as in deployed mode; only the launch differs. The steps
are the same as on the workshop site: **predict** the answer, **run** one cell, **check**
in the dashboard, **discuss**.

Every run is one `flwr run --run-config "..."` command: the cell prints it. Only the run
configuration changes from one step to the next, never the code.

| Dashboard tab | What it shows |
|---|---|
| **Solo or together?** | The federated model versus each client trained alone, and versus fully centralised training |
| **The run** | The local models in each round, who took part, client drift |
| **Network** | The messages exchanged during a round, replayed as an animation |
| **Compare** | Several runs on one chart |

A run takes 1 to 2 minutes; the first round takes 20 to 60 s, while the clients start.

## 0. Setup

In [ ]:
from flworkshop import viz

viz.setup()

Click **Open the dashboard** and keep the tab open next to this one: it updates every
round. Your client is your seat number minus one (clients are numbered from 0).

## 1. First run

Each client holds only **2 classes** out of 10: one has only T-shirts and sneakers,
another only trousers and shirts, and so on.

> **Predict.** How many of the 10 classes can a client recognise alone? How many will
> the federated model recognise after 10 rounds?

Then run, and open the **Solo or together?** tab. The "solo" and "centralised"
references were computed in advance with exactly the same data and training budget.

In [ ]:
viz.run("pathological", "partitioning='pathological'")

Now look at what travels on the network: the **Network** tab replays one round, and the server log shows the same exchanges.

In [ ]:
viz.open("pathological", view="network")
viz.log("pathological")

> **Discuss**
> 1. How many classes does a client recognise alone? And the federated model?
> 2. In *What each model can recognise*, two classes are recognised by nobody, not even
>    the centralised model. Why? (Look at *Each client's data*.)
> 3. Does a client ever talk to another client? In which order do the messages travel?
> 4. How many MB travelled in 10 rounds? Compare with the volume of the clients' data.

## 2. Heterogeneity

With **IID** data, each client holds a representative sample of all classes. With a
**Dirichlet** partitioning (α = 0.1), each client mostly holds a few classes, and some
clients hold far more images than others.

> **Predict.** Which run reaches the higher accuracy? Which one has the longer rounds,
> and why?

Run one cell after the other: a new run stops the one in progress.

In [ ]:
viz.run("iid", "partitioning='iid'")

In [ ]:
viz.run("dirichlet", "partitioning='dirichlet' dirichlet-alpha=0.1")

In [ ]:
viz.open("dirichlet", view="network")

> **Discuss**
> 1. *Compare* tab: rank the three runs. What makes federated learning hard?
> 2. *Network* tab, timeline of `dirichlet`: why does everyone wait for the same client?
>    Where have you seen this pattern on a cluster?
> 3. *The run* tab, *Client drift*: which client moves furthest from the global model?
>    What is special about its data?

## 3. Compute vs communication

Same compute budget, **20 local epochs per client**, split into more or fewer rounds.
Each round costs one exchange of models between the server and every client.

> **Predict.** With 2 rounds × 10 local epochs, the clients exchange 5 times fewer
> messages than in step 2. Does the accuracy drop on IID data? On Dirichlet data? Where
> does it drop more?

In [ ]:
viz.run("iid_r2_ep10", "partitioning='iid' num-server-rounds=2 local-epochs=10", view="compare")

In [ ]:
viz.run("dirichlet_r2_ep10", "partitioning='dirichlet' dirichlet-alpha=0.1 num-server-rounds=2 local-epochs=10", view="compare")

Complete the budget on Dirichlet data. In a group, split these two runs between you and
share the numbers (volume exchanged in the *Network* tab, final accuracy in *Compare*).

In [ ]:
viz.run("dirichlet_r20_ep1", "partitioning='dirichlet' dirichlet-alpha=0.1 num-server-rounds=20 local-epochs=1", view="compare")

In [ ]:
viz.run("dirichlet_r5_ep4", "partitioning='dirichlet' dirichlet-alpha=0.1 num-server-rounds=5 local-epochs=4", view="compare")

> **Discuss**
> 1. How does the volume exchanged scale with the number of rounds?
> 2. On IID data, what does it cost to talk 5 times less? And on Dirichlet data?
> 3. With many local epochs, each client drifts towards its own data before the next
>    average. Where have you met this trade-off between local work and synchronisation
>    in HPC?

## 4. Failures

In practice, only part of the clients train in each round, and some do not finish.

> **Predict.** If half of the clients drop out during a round, does the training stop?
> With only 2 clients out of 5 in each round, does the curve still go up smoothly?

**a.** Only 2 clients out of 5, sampled at random, train in each round:

In [ ]:
viz.run("participation_40", "partitioning='dirichlet' dirichlet-alpha=0.1 fraction-train=0.4", view="run")

**b.** Each client has a one-in-two chance of disconnecting during a round:

In [ ]:
viz.run("dropouts_50", "partitioning='dirichlet' dirichlet-alpha=0.1 dropout-rate=0.5", view="run")

> **Discuss**
> 1. Does the federation stop when a client drops out? What does the server do with the
>    missing ones? (*Network* tab on `dropouts_50`.)
> 2. Why does the global model curve go up in a saw-tooth? Look at who takes part in
>    each round and which classes they hold.
> 3. *Compare* tab with `dirichlet`: why is the last-round accuracy no longer enough to
>    compare two runs?

## Going further

**Harder images: CIFAR-10** (planes, cars, animals…), if the CIFAR-10 data is present:

In [ ]:
# viz.run("cifar", "dataset='cifar10' partitioning='pathological'")

**FedProx** penalises the distance between the local model and the global model
(`proximal-mu`). Does it beat FedAvg here? Look at *Client drift* in *The run* tab.

In [ ]:
# viz.run("fedprox", "partitioning='dirichlet' dirichlet-alpha=0.1 proximal-mu=0.1", view="run")

## Your turn

Every key below can go in the `--run-config` string; strings take single quotes.

| Key | Default | Role |
|---|---|---|
| `partitioning` | `'iid'` | `'iid'`, `'dirichlet'` or `'pathological'` |
| `dirichlet-alpha` | 0.1 | Smaller = more unbalanced |
| `classes-per-partition` | 2 | Classes per client with `'pathological'` |
| `fraction-train` | 1.0 | Share of clients sampled in each round |
| `dropout-rate` | 0.0 | Probability that a client drops out during a round |
| `num-server-rounds` | 10 | Number of rounds |
| `local-epochs` | 2 | Local training epochs per round |
| `learning-rate` | 0.01 | Client learning rate |
| `proximal-mu` | 0.0 | > 0 enables FedProx |
| `dataset` | `'fashion_mnist'` | `'fashion_mnist'` or `'cifar10'` |

The "solo / centralised" references exist only for the step 1 and 2 configurations;
elsewhere, the *Solo or together?* tab shows the federated model alone.
`viz.stop()` stops the run in progress.

In [ ]:
# viz.run("my_try", "partitioning='pathological' classes-per-partition=3")